# Quantização e Benchmark — Comparando fp32 vs int8 dinâmico

Extensão do case sobre [dgodevais/coursera-genai-for-data-science](https://github.com/dgodevais/coursera-genai-for-data-science).

## Decisão técnica

O notebook original (`quantization-demo.ipynb`) compara Mistral-7B em **float32, float16, 8-bit e 4-bit**
usando `BitsAndBytesConfig` — **todas as variantes 8/4-bit exigem GPU CUDA** (`bitsandbytes` não roda em CPU).

Aqui aplicamos **quantização dinâmica pós-treino** (`torch.quantization.quantize_dynamic`), que é
**nativa do PyTorch, roda 100% em CPU e é gratuita** — a técnica correta quando o alvo de deploy
não tem GPU (ex: servir o classificador numa Lambda/Cloud Run/container comum).

Mantemos a mesma estrutura de comparação do original: `print_size_of_model` + `check_prediction`,
rodando **antes e depois** da quantização, no **mesmo modelo fine-tunado no notebook anterior**.

In [ ]:
import os
import sys
import time

sys.path.append("../src")

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from data_prep import ID2LABEL

MODEL_PATH = "../models/distilbert-so-quality-ft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

## Funções auxiliares (mesmo nome/espírito do notebook original)

In [ ]:
def print_size_of_model(model, label=""):
    torch.save(model.state_dict(), "/tmp/temp_model.p")
    size_mb = os.path.getsize("/tmp/temp_model.p") / 1e6
    os.remove("/tmp/temp_model.p")
    print(f"{label}: {size_mb:.2f} MB")
    return size_mb


def check_prediction(model, tokenizer, text, label=""):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    model.eval()
    start = time.perf_counter()
    with torch.no_grad():
        logits = model(**inputs).logits
    elapsed_ms = (time.perf_counter() - start) * 1000
    pred_id = torch.argmax(logits, dim=-1).item()
    print(f"{label}: predição = {ID2LABEL[pred_id]}  |  latência = {elapsed_ms:.1f} ms")
    return pred_id, elapsed_ms


stackoverflow_question = (
    "Java: Repeat Task Every Random Seconds [SEP] "
    "I'm already familiar with repeating tasks every n seconds by using "
    "Java.util.Timer and Java.util.TimerTask. But lets say I want to print "
    "Hello World to the console every random seconds from 1-5. [SEP] tags: java"
)

## Modelo fine-tunado — float32 (baseline, sem quantização)

In [ ]:
model_fp32 = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

print("============ DistilBERT fine-tunado float32 (baseline) ==================")
size_fp32 = print_size_of_model(model_fp32, "DistilBERT float32")
pred_fp32, latency_fp32 = check_prediction(model_fp32, tokenizer, stackoverflow_question, "float32")

## Modelo fine-tunado — int8 (quantização dinâmica, pós-treino, CPU-native)

`quantize_dynamic` converte as camadas `Linear` (onde mora a maior parte dos parâmetros)
de float32 para int8 **só nos pesos** — as ativações continuam em float32 e são quantizadas
dinamicamente a cada inferência. É o tipo de quantização recomendado pra modelos baseados
em Linear/Transformer rodando em CPU.

In [ ]:
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32, {torch.nn.Linear}, dtype=torch.qint8
)

print("============ DistilBERT fine-tunado int8 (quantização dinâmica) ==================")
size_int8 = print_size_of_model(model_int8, "DistilBERT int8")
pred_int8, latency_int8 = check_prediction(model_int8, tokenizer, stackoverflow_question, "int8")

## Comparativo final

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "métrica": ["Tamanho em disco (MB)", "Latência de inferência (ms)", "Predição"],
    "float32 (baseline)": [f"{size_fp32:.1f}", f"{latency_fp32:.1f}", ID2LABEL[pred_fp32]],
    "int8 (quantizado)": [f"{size_int8:.1f}", f"{latency_int8:.1f}", ID2LABEL[pred_int8]],
})
reduction_pct = (1 - size_int8 / size_fp32) * 100
print(comparison.to_string(index=False))
print(f"\nRedução de tamanho: {reduction_pct:.1f}%")
print(f"Predição consistente entre versões: {pred_fp32 == pred_int8}")

## Conclusão

A quantização dinâmica reduz o tamanho do modelo em disco (tipicamente ~65-75% pra arquiteturas
baseadas em `Linear`, já que int8 usa 1/4 do espaço de float32) **sem precisar de GPU e sem
re-treinar** — só compressão pós-treino. Pra um cenário real de deploy (ex: classificador de
qualidade de post rodando atrás de uma API), isso significa menos memória, menos custo de
infraestrutura e cold-start mais rápido em ambientes serverless.

**Trade-off a observar:** quantização agressiva pode degradar a acurácia em modelos maiores/mais
sensíveis — a prática correta de produção é sempre revalidar as métricas do notebook 01
(accuracy, F1 macro) rodando o holdout de teste com `model_int8`, não só comparar uma predição isolada.
Deixado como próximo passo aqui de propósito, pra manter os dois notebooks focados.

---
*Case study construído via vibe-coding com Claude, sobre a base do curso Generative AI for Data Science (Microsoft/Coursera) — [Thiago Cassiano Gonçalves](https://github.com/ThiagoCassianoo).*